# Analyzing your sensor and survey data

This notebook walks through a basic analysis of the iMotions sensor data and Qualtrics survey data delivered by the lab. Run the cells from top to bottom (select a cell and press **Shift + Enter**). Only the cells in the **Settings** section normally need to be edited.

Steps:

1. Load the data
2. Build one dataset that combines survey answers and sensor measures
3. Plot the distribution of two variables (KDE plot)
4. Run an OLS regression
5. Run a SurvivalScale analysis for Likert (agree/disagree) questions

Your folder should contain `Survey.xlsx`, `Survey.csv` and the `sensorData` folder.

In [ ]:
# Run this cell once to install the required packages. Remove the leading # on the next line first.
# %pip install CBACABLab SurvivalScale statsmodels

In [6]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

for project_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = project_root / "src"
    if (src_dir / "CBACABLab").is_dir():
        sys.path.insert(0, str(src_dir))
        break

## Settings

Change these to match your study. The question names (like `Q1`) are the question ids from your Qualtrics survey.

In [13]:
import pandas as pd
import CBACABLab

DATABASE = "data.db"          # file created to store your data
SURVEY_FILE = "Survey.xlsx"   # the Excel version of the survey (Survey.csv must sit beside it)
SENSOR_FOLDER = "./sensorData/"

# Questions at the respondent level (e.g. demographics, personality, etc.) that should not be connected to instument/sensor data.
RESPONDENT_QUESTIONS = ["Q4_4", "Q5_4", "Q6", "Q7"]

# Likert questions (e.g. Strongly disagree ... Strongly agree) and the number each answer maps to
AGREEMENT_SCALE = {
    "Strongly disagree": 0,
    "Somewhat disagree": 1,
    "Neither agree nor disagree": 2,
    "Somewhat agree": 3,
    "Strongly agree": 4,
}
LIKERT_SCALES = {question: AGREEMENT_SCALE for question in ["Q1", "Q2", "Q3"]}

# Screen regions (tags) shown to participants, and the sensor measures to include
TAGS = ["left", "right"]
INSTRUMENT_PROPERTIES = ["Attention", "Joy"]

## 1. Load the data

This reads the survey and sensor files into a local database (`data.db`). It can take a few minutes. Each region of the screen is loaded as a set of bounding boxes `(left, right, top, bottom)` (both have only one bounded box but more are supported for each tag) in pixels with its own tag.  This isn't necessary for all analyses, but it allows you to analyze the data by region of the screen.  If you don't need this, you can set `boundingBoxes = None` in the `LoadSensorData` call below.  You would also only need to load the sensor data once, and then you can reuse the database for future analyses.



In [11]:
# Matches survey question ids to the names used in the sensor files
question_mapping = CBACABLab.AutoDetermineQuestionIdtoQuestionNameMapping(SURVEY_FILE, surveyCSVName="Survey.csv")

CBACABLab.LoadSurveyData(databaseName=DATABASE, surveyName=SURVEY_FILE)

# Left half of a 1920x1080 screen. rebuild=True clears old sensor data first.
CBACABLab.LoadSensorData(
   databaseName=DATABASE,
    sensorFolder=SENSOR_FOLDER,
    boundedboxes=[(0, 960, 0, 1080)],
    tag="left",
    rebuild=True,
    questionIdtoQuestionName=question_mapping,
)

# Right half of the screen
CBACABLab.LoadSensorData(
    databaseName=DATABASE,
    sensorFolder=SENSOR_FOLDER,
    boundedboxes=[(960, 1920, 0, 1080)],
    tag="right",
    questionIdtoQuestionName=question_mapping,
)

# See which survey questions are available
CBACABLab.ListQuestions(DATABASE)

['Last Seen Flow Element ID',
 'Q1',
 'Q2',
 'Q3',
 'Q4_4',
 'Q5_4',
 'Q6',
 'Q7',
 'iMotionsRespondent',
 'iMotionsStudy']

## 2. Build the analysis dataset

This combines the survey answers and sensor measures into one table, with one row for each person and Likert question.

In [ ]:
panel = CBACABLab.CreatePanelData(
    question_level_columns=list(LIKERT_SCALES),
    respondent_level_columns=RESPONDENT_QUESTIONS,
    tagFilter=TAGS,
    instrumentProperties=INSTRUMENT_PROPERTIES,
    databaseName=DATABASE,
)
panel.head(10)

,respondent_id,question,response,duration_left,duration_right,pupil_left,pupil_right,Attention_left,Attention_right,Joy_left,Joy_right,Q4_4,Q5_4,Q6,Q7
0,1,Q1,Strongly disagree,343.472949,357.075008,2.628579,2.714873,98.137547,97.839279,0.034117,0.037362,65.198,22.066,Other,Bachelor's Degree
1,1,Q2,Neither agree nor disagree,358.868249,1916.269888,2.776471,2.833336,97.348034,98.041817,0.120167,1.596050,65.198,22.066,Other,Bachelor's Degree
2,1,Q3,Strongly agree,409.204696,694.727733,2.817981,2.836787,NaN,97.550504,NaN,0.052147,65.198,22.066,Other,Bachelor's Degree
3,2,Q1,Strongly agree,384.045313,722.940208,2.693345,2.841391,97.755627,97.696284,0.025384,0.025649,89.207,94.341,White/ Caucasian,Prefer not to say
4,2,Q2,Neither agree nor disagree,1027.143104,398.548022,2.739849,2.741312,95.377090,97.866264,0.025275,0.024139,89.207,94.341,White/ Caucasian,Prefer not to say
5,2,Q3,Somewhat disagree,736.664201,NaN,2.752320,NaN,97.902746,NaN,25.002819,NaN,89.207,94.341,White/ Caucasian,Prefer not to say
6,3,Q1,Neither agree nor disagree,380.960150,264.400428,2.719926,2.847198,98.039979,98.498688,26.620419,0.036503,100.000,2.458,White/ Caucasian,High school diploma / GED
7,3,Q2,Strongly disagree,591.957592,438.112677,2.702798,2.532557,97.798824,98.661983,90.557137,0.024104,100.000,2.458,White/ Caucasian,High school diploma / GED
8,3,Q3,Somewhat disagree,733.998717,257.911149,2.662162,2.658208,97.131732,98.370094,0.029900,0.038205,100.000,2.458,White/ Caucasian,High school diploma / GED
9,4,Q1,Somewhat agree,385.729328,402.003738,2.624093,2.658857,97.832081,NaN,0.043469,NaN,51.432,7.629,Asian/ Asian American,Master's Degree


In [ ]:
# The names of all columns you can use in the steps below
list(panel.columns)

## 3. Plot two variables (KDE plot)

A KDE plot is a smooth version of a histogram that shows how a variable's values are spread out. Here we compare the pupil measure for the left and right regions. Replace the column names with any numeric columns listed above. Each person appears once per question, so keep one row per person first to avoid counting them more than once.

In [ ]:
PLOT_COLUMNS = ["pupil_left", "pupil_right"]

people = panel.drop_duplicates("respondent_id")
series = [pd.to_numeric(people[column], errors="coerce").dropna() for column in PLOT_COLUMNS]

CBACABLab.ProjectKDEPlot(
    series,
    label="Pupil measure",
    labels=["Left", "Right"],
    mean=True,   # draw a line at each average
)

## 4. OLS regression

OLS (ordinary least squares) is used when the outcome is a number, such as a slider answer or a sensor measure. Here the outcome is `pupil_left`, explained by which question was asked. Because each person answers several questions, the standard errors are clustered by person.

First the Likert answers are converted to numbers (`k`) and `bound` (the number of points on the scale). Answers that are not on the scale, such as "Don't know", are removed.

In [ ]:
# Keep only answers that appear on the question's scale
valid = [
    str(response).strip().lower() in {str(label).strip().lower() for label in LIKERT_SCALES[question]}
    for question, response in zip(panel["question"], panel["response"])
]
panel_likert = CBACABLab.AddLikertColumns(panel[valid], LIKERT_SCALES).reset_index(drop=True)

# The columns the analyses need: id, block (who to cluster by), question, the variable(s) of interest, k and bound
FEATURES = ["pupil_left"]
analysis_data = pd.DataFrame({
    "id": panel_likert["respondent_id"],
    "block": panel_likert["respondent_id"],
    "question": panel_likert["question"],
    "pupil_left": pd.to_numeric(panel_likert["pupil_left"], errors="coerce"),
    "k": panel_likert["k"].astype(int),
    "bound": panel_likert["bound"].astype(int),
}).dropna()
analysis_data.head(10)

In [ ]:
import statsmodels.formula.api as smf

# Formula: outcome ~ explanatory variables. C(question) treats each question as a category.
ols_model = smf.ols("pupil_left ~ C(question)", data=analysis_data).fit(
    cov_type="cluster", cov_kwds={"groups": analysis_data["block"]}
)
print(ols_model.summary())

## 5. SurvivalScale for Likert responses

Likert answers are ordered categories, so OLS isn't ideal for them. [SurvivalScale](https://github.com/tazzben/Likert-estimation) estimates how the variables in `FEATURES` affect the Likert response, using the `k` and `bound` columns prepared above. The bootstrap repeats the estimate many times to calculate confidence intervals; increase `bootstrap_iterations` (for example to 1000) for final results, at the cost of a longer run.

In [ ]:
from SurvivalScale import get_results

cj, beta, metrics = get_results(
    analysis_data,
    bootstrap_iterations=100,
    alpha=0.05,          # 95% confidence intervals
    columns=FEATURES,
    block_id="block",
)
display(beta)     # effect of each variable in FEATURES
display(metrics)  # overall model fit